# Notebook 05 — Model Dimensions, Bounds, and Theta Layout

This notebook constructs `ModelDims`, builds the optimisation bounds,
explains the theta vector layout, and demonstrates `decode_theta()`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from phoscrosstalk.config import ModelDims
from phoscrosstalk.data_loader import load_site_data, load_kinase_site_matrix
from phoscrosstalk.optimization import create_bounds, build_parameter_labels
from phoscrosstalk.mechanisms import decode_theta

timepoints = list(range(1, 15))
sites, proteins, site_prot_idx, positions, t, Y, A_data, A_proteins = \
    load_site_data(SAMPLE_DIR / "protephospho.csv", timepoints)
K_site_kin, kinases = load_kinase_site_matrix(SAMPLE_DIR / "kinase_sites.tsv", sites)

K = len(proteins)
M = len(kinases)
N = len(sites)

dims = ModelDims.set_dims(K, M, N)
print(f"ModelDims: K={K}  M={M}  N={N}")
print(f"Theta dimension = 2K + 2 + 3M + N + 4 = {2*K + 2 + 3*M + N + 4}")


## 1 · Theta vector layout

In [ ]:
dim = 2*K + 2 + 3*M + N + 4
print(f"theta ∈ R^{dim}")
print()

blocks = [
    ("log_k_deact",  K,  0,    "protein deactivation rates (log-space)"),
    ("log_d_deg",    K,  K,    "protein degradation rates (log-space)"),
    ("log_beta_g",   1,  2*K,  "global PTM crosstalk coupling (log-space)"),
    ("log_beta_l",   1,  2*K+1,"local sequence crosstalk coupling (log-space)"),
    ("log_alpha",    M,  2*K+2,"kinase activation strengths (log-space)"),
    ("log_kK_act",   M,  2*K+2+M,  "kinase activation rates (log-space)"),
    ("log_kK_deact", M,  2*K+2+2*M,"kinase deactivation rates (log-space)"),
    ("log_k_off",    N,  2*K+2+3*M,"phosphatase dephosphorylation rates (log-space)"),
    ("raw_gamma",    4,  2*K+2+3*M+N,"signed regulatory coupling (tanh-encoded)"),
]

print(f"{'Block':<16} {'Length':>7}  {'Start':>6}  {'End':>6}  Description")
print("-"*72)
end = 0
for name, length, start, desc in blocks:
    end = start + length
    print(f"{name:<16} {length:>7}  {start:>6}  {end:>6}  {desc}")
print(f"{'TOTAL':<16} {dim:>7}")


## 2 · create_bounds()

In [ ]:
xl, xu, dim = create_bounds(K, M, N)
print(f"xl shape: {xl.shape}  (lower bounds in log-space)")
print(f"xu shape: {xu.shape}  (upper bounds in log-space)")
print(f"dim = {dim}")
print()
print("First 10 bounds (log-space):")
for i in range(min(10, dim)):
    print(f"  [{i:2d}]  xl={xl[i]:7.3f}  xu={xu[i]:7.3f}")


## 3 · Parameter labels DataFrame

In [ ]:
labels = build_parameter_labels(K, M, N)

# Convert log-space bounds back to original scale for readability
xl_orig = np.exp(xl)
xu_orig = np.exp(xu)
# gamma parameters are NOT log-space (last 4)
xl_orig[-4:] = xl[-4:]
xu_orig[-4:] = xu[-4:]

df_bounds = pd.DataFrame({
    "parameter": labels,
    "xl (log-space)": xl.round(4),
    "xu (log-space)": xu.round(4),
    "xl (original scale)": xl_orig.round(6),
    "xu (original scale)": xu_orig.round(4),
})
print(df_bounds.to_string(index=False))


## 4 · Log-space encoding

In [ ]:
# Demonstrate why log-space encoding is used
rates = np.logspace(-5, 1, 200)
log_rates = np.log(rates)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(rates, np.ones_like(rates), "|", markersize=6)
axes[0].set_xscale("log")
axes[0].set_xlabel("rate (original scale)")
axes[0].set_title("Rate values span 6 orders of magnitude")
axes[0].set_yticks([])

axes[1].plot(log_rates, np.ones_like(log_rates), "|", markersize=6, color="orange")
axes[1].set_xlabel("log(rate)")
axes[1].set_title("Log-space: uniform distribution over optimisation range")
axes[1].set_yticks([])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_log_space_encoding.png", dpi=100)
plt.show()


## 5 · decode_theta() demonstration

In [ ]:
# Sample a random theta within bounds
rng = np.random.default_rng(42)
theta_rand = rng.uniform(xl, xu)

(k_deact, d_deg,
 beta_g, beta_l,
 alpha, kK_act, kK_deact,
 k_off,
 gamma_S_p, gamma_A_S, gamma_A_p, gamma_K_net) = decode_theta(theta_rand, K, M, N)

import jax.numpy as jnp

print("Decoded parameters from a random theta:")
print(f"  k_deact   : {np.array(k_deact).round(5)}  shape ({K},)  — protein deactivation rates")
print(f"  d_deg     : {np.array(d_deg).round(5)}  shape ({K},)  — protein degradation rates")
print(f"  beta_g    : {float(beta_g):.5f}  (scalar)  — global crosstalk coupling")
print(f"  beta_l    : {float(beta_l):.5f}  (scalar)  — local crosstalk coupling")
print(f"  alpha     : {np.array(alpha).round(5)}  shape ({M},)  — kinase strengths")
print(f"  kK_act    : {np.array(kK_act).round(5)}  shape ({M},)  — kinase activation rates")
print(f"  kK_deact  : {np.array(kK_deact).round(5)}  shape ({M},)  — kinase deactivation rates")
print(f"  k_off     : {np.array(k_off).round(5)}  shape ({N},)  — phosphatase rates")
print(f"  gamma_S_p : {float(gamma_S_p):.5f}  — phosphosite→protein coupling")
print(f"  gamma_A_S : {float(gamma_A_S):.5f}  — protein→phosphosite feedback")
print(f"  gamma_A_p : {float(gamma_A_p):.5f}  — (compatibility)")
print(f"  gamma_K_net:{float(gamma_K_net):.5f}  — kinase network coupling")


## 6 · Gamma encoding: tanh → [-2, +2]

The four `gamma` parameters are **not** log-encoded.  They use `2·tanh(raw_gamma)`,
yielding a continuous range of ≈ (−2, +2) that represents signed regulatory coupling.


In [ ]:
raw = np.linspace(-3, 3, 300)
gamma = 2 * np.tanh(raw)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(raw, gamma)
ax.axhline(2,  color="r", linestyle="--", alpha=0.5, label="±2 asymptote")
ax.axhline(-2, color="r", linestyle="--", alpha=0.5)
ax.axhline(0,  color="k", linestyle=":", alpha=0.3)
ax.set_xlabel("raw_gamma (optimiser variable)")
ax.set_ylabel("gamma = 2·tanh(raw)")
ax.set_title("Gamma encoding: sigmoid-bounded regulatory coupling")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_gamma_encoding.png", dpi=100)
plt.show()


## 7 · ODE state vector

In [ ]:
print("ODE state vector  y = [R_rna(K), S(K), A(K), Kdyn(M), p(N)]")
print(f"  R_rna : ({K},)  mRNA-driven protein synthesis rates")
print(f"  S     : ({K},)  protein signalling active fraction")
print(f"  A     : ({K},)  protein abundance")
print(f"  Kdyn  : ({M},)  kinase dynamic activation state")
print(f"  p     : ({N},)  phosphosite occupancy")
print()
total_state_dim = 3*K + M + N
print(f"Total ODE dimension = 3K + M + N = 3·{K} + {M} + {N} = {total_state_dim}")
